<a href="https://colab.research.google.com/github/Riana-PSB/Hello/blob/main/AIP_Mini_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math

TARGET_SCORE = 20
MAX_TURNS = 10

DICE_OUTCOMES = [
    (-3, 1/6, False),
    (1, 2/6, False),
    (4, 2/6, False),
    (6, 1/6, True)
]

class GameState:
    def __init__(self, ai_score, opp_score, turn, is_ai_turn):
        self.ai_score = ai_score
        self.opp_score = opp_score
        self.turn = turn
        self.is_ai_turn = is_ai_turn

    def is_terminal(self):
        return (
            self.ai_score >= TARGET_SCORE or
            self.opp_score >= TARGET_SCORE or
            self.turn > MAX_TURNS
        )

    def evaluate(self):
        return self.ai_score - self.opp_score


# 🎲 Expectiminimax
def expectiminimax(state, depth):
    if state.is_terminal() or depth == 0:
        return state.evaluate()

    if state.is_ai_turn:
        return max_value(state, depth)
    else:
        return min_value(state, depth)


def max_value(state, depth):
    safe = GameState(state.ai_score + 2, state.opp_score, state.turn + 1, False)
    safe_val = expectiminimax(safe, depth - 1)

    risky_val = chance_node(state, depth, True)

    return max(safe_val, risky_val)


def min_value(state, depth):
    safe = GameState(state.ai_score, state.opp_score + 2, state.turn + 1, True)
    safe_val = expectiminimax(safe, depth - 1)

    risky_val = chance_node(state, depth, False)

    return min(safe_val, risky_val)


def chance_node(state, depth, is_ai):
    total = 0

    for value, prob, extra_turn in DICE_OUTCOMES:
        if is_ai:
            ai = state.ai_score + value
            opp = state.opp_score
            next_ai = extra_turn
        else:
            ai = state.ai_score
            opp = state.opp_score + value
            next_ai = not extra_turn

        turn = state.turn if extra_turn else state.turn + 1

        next_state = GameState(ai, opp, turn, next_ai)

        total += prob * expectiminimax(next_state, depth - 1)

    return total


def best_move(state, depth=4):
    safe = GameState(state.ai_score + 2, state.opp_score, state.turn + 1, False)
    safe_val = expectiminimax(safe, depth - 1)

    risky_val = chance_node(state, depth, True)

    if risky_val > safe_val:
        return "risky"
    return "safe"


# 🎮 Roll dice
def roll_dice():
    r = random.randint(1, 6)

    if r == 1:
        return -3, False, r
    elif r in [2, 3]:
        return 1, False, r
    elif r in [4, 5]:
        return 4, False, r
    else:
        return 6, True, r


# 🕹️ GAME LOOP
def play_game():
    state = GameState(0, 0, 1, True)

    print("🎲 Treasure Dice Duel 🎲")
    print("You vs AI\n")

    while not state.is_terminal():
        print(f"\nTurn {state.turn}")
        print(f"AI: {state.ai_score} | You: {state.opp_score}")

        if state.is_ai_turn:
            print("\n🤖 AI thinking...")
            move = best_move(state)

            print(f"AI chooses: {move}")

            if move == "safe":
                state.ai_score += 2
                state.turn += 1
                state.is_ai_turn = False
            else:
                value, extra, roll = roll_dice()
                print(f"AI rolled {roll} → {value} points")

                state.ai_score += value

                if not extra:
                    state.turn += 1
                    state.is_ai_turn = False
                else:
                    print("AI gets extra turn!")

        else:
            print("\nYour move:")
            choice = input("Choose (safe/risky): ").lower()

            if choice == "safe":
                state.opp_score += 2
                state.turn += 1
                state.is_ai_turn = True

            elif choice == "risky":
                value, extra, roll = roll_dice()
                print(f"You rolled {roll} → {value} points")

                state.opp_score += value

                if not extra:
                    state.turn += 1
                    state.is_ai_turn = True
                else:
                    print("You get extra turn!")

            else:
                print("Invalid input, try again.")

    # 🏁 Game Over
    print("\n🏁 Game Over!")
    print(f"Final Score → AI: {state.ai_score} | You: {state.opp_score}")

    if state.ai_score > state.opp_score:
        print("🤖 AI Wins!")
    elif state.ai_score < state.opp_score:
        print("🎉 You Win!")
    else:
        print("It's a Draw!")

def simulate_game():
    state = GameState(0, 0, 1, True)

    while not state.is_terminal():
        if state.is_ai_turn:
            move = best_move(state)

            if move == "safe":
                state.ai_score += 2
                state.turn += 1
                state.is_ai_turn = False
            else:
                value, extra, _ = roll_dice()
                state.ai_score += value

                if not extra:
                    state.turn += 1
                    state.is_ai_turn = False

        else:
            # Opponent ALSO uses AI (minimizing player)
            move = best_move(state)

            if move == "safe":
                state.opp_score += 2
                state.turn += 1
                state.is_ai_turn = True
            else:
                value, extra, _ = roll_dice()
                state.opp_score += value

                if not extra:
                    state.turn += 1
                    state.is_ai_turn = True

    # Return result
    if state.ai_score > state.opp_score:
        return "AI"
    elif state.ai_score < state.opp_score:
        return "Opponent"
    else:
        return "Draw"


def run_simulation(n=100):
    results = {"AI": 0, "Opponent": 0, "Draw": 0}

    for i in range(n):
        winner = simulate_game()
        results[winner] += 1

    print("\n📊 Simulation Results")
    print(f"Total Games: {n}")
    print(f"AI Wins: {results['AI']} ({results['AI']/n*100:.1f}%)")
    print(f"Opponent Wins: {results['Opponent']} ({results['Opponent']/n*100:.1f}%)")
    print(f"Draws: {results['Draw']} ({results['Draw']/n*100:.1f}%)")


# ▶️ Run simulation instead of play_game()
if __name__ == "__main__":
    #run_simulation(100)
    play_game()

